# 🧩 이미지 슬라이드 → 편집 가능한 PPTX (slide-unflatten)

NotebookLM·Gemini·Canva·Gamma 등에서 받은 **통이미지 슬라이드(PPTX/PDF/PNG)** 를 글자와 그림을 고칠 수 있는 PPTX로 바꿉니다.

## 🙋 처음 한 번만
1. **파일 → 드라이브에 사본 저장** 후 사본에서 작업하세요.
2. **런타임 → 런타임 유형 변경 → T4 GPU → 저장** (없어도 되지만 느립니다).
3. *(선택, 강력 추천)* Claude API 키: [console.anthropic.com](https://console.anthropic.com) → Billing 충전 → API Keys → 키 복사 → 왼쪽 **🔑 보안 비밀**에 이름 `ANTHROPIC_API_KEY`, 값 = 키, **노트북 액세스** 켜기.
   키가 있으면 오타 교정·문단 묶기·그림 개체 분리까지 됩니다. 슬라이드 1장당 약 60원(Opus) / 30원(Sonnet).

## ▶️ 매번
**런타임 → 모두 실행** → 4번 셀에서 파일 선택 → 끝나면 `*_editable.pptx`와 리포트가 자동 다운로드됩니다.
웹 화면이 편하면 맨 아래 **웹 UI** 셀을 실행하세요.

> 🔒 파일은 내 Colab 작업 공간에서 처리됩니다. Claude를 쓰면 슬라이드 이미지가 Anthropic으로 전송됩니다.

## 1. 설치 (약 2분)

In [ ]:
#@title 설치 { display-mode: "form" }
REPO_URL = "https://github.com/Leewanglyul/slide-unflatten.git"  #@param {type:"string"}
import subprocess, sys, importlib
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "-q", "install", *a], check=False)
pip(f"slide-unflatten[claude,ocr,segment,ui] @ git+{REPO_URL}")
pip("--no-deps", "simple-lama-inpainting")          # 옛 Pillow 핀 때문에 --no-deps
subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum", "fonts-noto-cjk"], capture_output=True)
try:
    importlib.import_module("unflatten")
except ImportError:
    print("⚠️ git 설치 실패. 패키지 zip을 올려 주세요 (slide-unflatten 폴더를 zip으로).")
    from google.colab import files
    up = files.upload()
    for n in up:
        subprocess.run(["unzip", "-oq", n]); pip("-e", n[:-4] + "[claude,ocr,segment,ui]")
from unflatten.cli import doctor
doctor()

## 2. Claude API 키 확인
`API key: ✅`가 나오면 정상입니다. 없어도 규칙 기반으로 계속 진행됩니다.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = (userdata.get("ANTHROPIC_API_KEY") or "").strip().strip('"').strip("'")
except Exception as e:
    print("보안 비밀을 읽지 못했습니다:", e)
from unflatten.config import Config
from unflatten.claude_client import ClaudeService
USE_CLAUDE = False
if os.environ.get("ANTHROPIC_API_KEY"):
    ok, msg = ClaudeService(Config()).validate()
    USE_CLAUDE = ok
    print("API key:", "✅" if ok else "❌", msg)
else:
    print("API 키 없음 → 규칙 기반으로 진행합니다 (보안 비밀 이름 ANTHROPIC_API_KEY · 노트북 액세스 확인)")

## 3. 설정

In [ ]:
#@title ⚙️ 변환 설정 { display-mode: "form" }
MODE = "full"  #@param ["text", "objects", "full"] {type:"string"}
#@markdown `text`: 텍스트만(빠름·저렴) · `objects`: 텍스트+그림 통째 · `full`: 그림 속 개체까지 분리
LANGUAGE = "ko"  #@param ["ko", "en", "ja", "zh"]
MODEL = "claude-opus-5-5"  #@param ["claude-opus-5-5", "claude-sonnet-5-5"]
SLIDES = ""  #@param {type:"string"}
#@markdown 슬라이드 범위 (예: `1-3,7`). 비우면 전체.
KEEP_HIDDEN_ORIGINAL = True  #@param {type:"boolean"}
GROUP_SHAPES = True  #@param {type:"boolean"}
THEME_FONTS = True  #@param {type:"boolean"}
REMOVE_WATERMARK = True  #@param {type:"boolean"}
UPSCALE = 1  #@param [1, 2] {type:"raw"}
UPLOAD_FONTS = True  #@param {type:"boolean"}
#@markdown 글꼴 파일(.ttf/.otf) 업로드 창을 띄울지 (보통+굵게 두 개 권장, 취소하면 언어 기본 글꼴)

from unflatten.config import Config
from unflatten.fonts import assign_font_files, font_family
cfg = Config(mode=MODE, language=LANGUAGE, model=MODEL, use_claude=USE_CLAUDE, slide_range=SLIDES or None,
             keep_hidden_original=KEEP_HIDDEN_ORIGINAL, group_shapes=GROUP_SHAPES, use_theme_fonts=THEME_FONTS,
             remove_watermark=REMOVE_WATERMARK, upscale=int(UPSCALE), out_dir="/content/unflatten_out")
if UPLOAD_FONTS:
    from google.colab import files
    print("글꼴 파일을 선택하세요 (건너뛰려면 취소)")
    try:
        up = files.upload()
    except Exception:
        up = {}
    reg, bold = assign_font_files(list(up))
    cfg.font_regular_file, cfg.font_bold_file = reg, bold
    if reg:
        print("본문 글꼴:", font_family(reg), "| 제목 글꼴:", font_family(bold) if bold else "(본문 + 굵게)")
print("설정 완료:", cfg.mode, cfg.language, cfg.model if cfg.use_claude else "Claude 없음")

## 4. 파일 올리고 변환하기 (PPTX · PDF · PNG/JPG, 여러 개 가능)

In [ ]:
from google.colab import files
from unflatten.pipeline import convert, estimate, bundle_zip
from unflatten.cli import print_summary
uploaded = files.upload()
for name in uploaded:
    print("=====", name, "|", estimate(name, cfg)["line"])
results = []
for name in uploaded:
    try:
        res = convert(name, cfg)
        print_summary(res)
        results.append(res)
    except Exception as e:
        print("❌", name, type(e).__name__, e)
for res in results:
    files.download(res.out_path)
    if res.report_path: files.download(res.report_path)
    if res.font_files: files.download(bundle_zip(res))

## 5. 결과 확인 · 수정 · 재조립
- `*_report.html`: 슬라이드별 상태와 원본/결과 비교. 경고가 있는 장을 먼저 보세요.
- 글꼴만 바꿔 다시 만들려면 3번 셀에서 글꼴을 다시 올린 뒤 아래 셀을 실행하세요. 인식 결과는 캐시되어 몇 초면 끝납니다.

In [ ]:
from unflatten.pipeline import rebuild
from unflatten.cli import print_summary
for res in results:
    r2 = rebuild(res.manifest_path, {"font_regular_file": cfg.font_regular_file, "font_bold_file": cfg.font_bold_file,
                                     "group_shapes": cfg.group_shapes, "use_theme_fonts": cfg.use_theme_fonts,
                                     "keep_hidden_original": cfg.keep_hidden_original})
    print_summary(r2); files.download(r2.out_path)

## (선택) 웹 UI
업로드 → 옵션 → 예상 비용 → 변환 → 미리보기 → 다운로드가 한 화면에 있습니다. 텍스트 수정·재조립 탭도 있습니다.

In [ ]:
from unflatten.app import launch
launch(share=True, inline=True, ui_lang="ko")